In [28]:
import requests
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


In [29]:
@tool
def multiply(a:int, b:int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a*b

print(multiply.invoke({"a":2, "b":4}))
print(multiply.name)
print(multiply.description)
print(multiply.args)


8
multiply
Given 2 numbers a and b this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [30]:
llm = ChatOpenAI(model = "gpt-5.4-nano")
llm.invoke("hi")

AIMessage(content='Hi! 👋 How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 7, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--019ec734-8166-7b63-b547-0871b7509dbf-0', usage_metadata={'input_tokens': 7, 'output_tokens': 14, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [31]:
## tool binding
llm_with_tool = llm.bind_tools([multiply])
llm_with_tool.invoke("How are you?") ## here the llm is not using any tools. because the query is not related to the tools that's why it will not use tools.

AIMessage(content='I’m doing well—thanks for asking! How are you doing today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 138, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--019ec734-8c76-7543-91d6-c825a4d88bcd-0', usage_metadata={'input_tokens': 138, 'output_tokens': 18, 'total_tokens': 156, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [32]:
query = HumanMessage('can you multiply 3 with 1000')
messages = [query]
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [33]:
result = llm_with_tool.invoke(messages) #using tools
messages.append(result)
messages


[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_9pMOf7CFYnB1Mt0lTa0uspeL', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 143, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019ec734-9084-7b43-b85c-c7a551ebd2e9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_9pMOf7CFYnB1Mt0lTa0uspeL', 'type': 'tool_call'}], usage_metadata={'input_tokens': 143, 'output_tokens': 21, 'total_tokens': 164, 'input_token_det

In [34]:
tool_result = multiply.invoke(result.tool_calls[0])
print(tool_result)
messages.append(tool_result)

content='3000' name='multiply' tool_call_id='call_9pMOf7CFYnB1Mt0lTa0uspeL'


In [35]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_9pMOf7CFYnB1Mt0lTa0uspeL', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 143, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019ec734-9084-7b43-b85c-c7a551ebd2e9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_9pMOf7CFYnB1Mt0lTa0uspeL', 'type': 'tool_call'}], usage_metadata={'input_tokens': 143, 'output_tokens': 21, 'total_tokens': 164, 'input_token_det

In [36]:
llm_with_tool.invoke(messages).content

'3 × 1000 = **3000**.'

## Dollar to BDT exchange rate

In [100]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated


@tool
def get_conversion_factor(base_currency:str, target_currency:str)->float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f"https://v6.exchangerate-api.com/v6/48bc5bbc6ef5314561f73f99/pair/{base_currency}/{target_currency}"

  response = requests.get(url)

  return response.json()


@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """
  return base_currency_value* conversion_rate

In [101]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [102]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'BDT'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1781395201,
 'time_last_update_utc': 'Sun, 14 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1781481601,
 'time_next_update_utc': 'Mon, 15 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'BDT',
 'conversion_rate': 122.9387}

In [103]:
convert.invoke({'base_currency_value':10, 'conversion_rate':122.16})

1221.6

In [115]:
# tool binding
llm = ChatOpenAI(model = "gpt-4o-mini")

In [116]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [135]:
messages = [HumanMessage('What is the conversion factor between USD and BDT, and based on that can you convert 100 usd to bdt')]

In [136]:
messages

[HumanMessage(content='What is the conversion factor between USD and BDT, and based on that can you convert 100 usd to bdt', additional_kwargs={}, response_metadata={})]

In [137]:
ai_message = llm_with_tools.invoke(messages)

In [138]:
messages.append(ai_message)

In [139]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'BDT'},
  'id': 'call_fDWxLdzv1S0UWRdr77DXGut3',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 100},
  'id': 'call_dzYStGAxOTNTE66gXXkJAu4b',
  'type': 'tool_call'}]

In [140]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [141]:
messages

[HumanMessage(content='What is the conversion factor between USD and BDT, and based on that can you convert 100 usd to bdt', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_fDWxLdzv1S0UWRdr77DXGut3', 'function': {'arguments': '{"base_currency": "USD", "target_currency": "BDT"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'call_dzYStGAxOTNTE66gXXkJAu4b', 'function': {'arguments': '{"base_currency_value": 100}', 'name': 'convert'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 120, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0004d2b74a', 'finish_reason': 'tool_calls', 'logprobs': None}, id

In [142]:
llm_with_tools.invoke(messages)

AIMessage(content='The conversion factor between USD (United States Dollar) and BDT (Bangladeshi Taka) is approximately **122.9387**.\n\nBased on this conversion rate, converting 100 USD to BDT results in approximately **12,293.87 BDT**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 341, 'total_tokens': 398, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0004d2b74a', 'finish_reason': 'stop', 'logprobs': None}, id='run--019ec745-1452-7361-ac1d-2496202bd985-0', usage_metadata={'input_tokens': 341, 'output_tokens': 57, 'total_tokens': 398, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [143]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and BDT is approximately 122.9387. Therefore, converting 100 USD to BDT results in approximately 12,293.87 BDT.'

In [144]:
from langchain.agents import initialize_agent, AgentType

# Step 5: Initialize the Agent ---
agent_executor = initialize_agent(
    tools=[get_conversion_factor, convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,  # using ReAct pattern
    verbose=True  # shows internal thinking
)

In [145]:
# --- Step 6: Run the Agent ---
user_query = "Hi how are you?"

response = agent_executor.invoke({"input": user_query})



> Entering new AgentExecutor chain...
Thought: The user is greeting me and may be looking for a friendly response. 
Action:
```
{
  "action": "Final Answer",
  "action_input": "I'm just a computer program, but I'm here and ready to help you! How can I assist you today?"
}
```


> Finished chain.
